# Automatización de Infraestructura de Red con Python
## DevNet Sandbox — IOS XE Catalyst 8000v

---

### Topología

```
[Tu máquina]
     |
VPN AnyConnect
     |
     +──────────────────────+──────────────────────+
     |                      |                      |
[10.10.20.40]        [10.10.20.48]          [10.10.20.50]
Switch Nexus 9K      Cat8k-1 (R1)           DevBox Ubuntu
SSH directo          SSH directo                  |
                                         telnet localhost:2223
                                                  |
                                            Cat8k-2 (R2)
                                            cat8k-pod02
```

| Dispositivo | IP | Acceso | Usuario | Contraseña |
|---|---|---|---|---|
| Switch Nexus 9K | `10.10.20.40` | SSH directo | `admin` | `RG!_Yw200` |
| Router 1 (Cat8k-1) | `10.10.20.48` | SSH directo | `developer` | `C1sco12345` |
| DevBox | `10.10.20.50` | SSH directo | `developer` | `C1sco12345` |
| Router 2 (Cat8k-2) | via DevBox | telnet :2223 | `developer` | `C1sco12345` |

> **REQUISITO:** Conectar el VPN AnyConnect del sandbox antes de ejecutar las celdas de configuración.

---
## Celda 1 — Instalación de librerías

Ejecutar esta celda una sola vez. En Colab las librerías se reinstalan en cada sesión.

In [ ]:
# Instalar las librerías necesarias
# - netmiko: abstracción SSH para dispositivos Cisco
# - paramiko: SSH de bajo nivel para el túnel al DevBox

%pip install netmiko paramiko

print("Librerías instaladas correctamente.")

---
## Celda 2 — Imports

In [ ]:
from netmiko import ConnectHandler
import paramiko
import time
import sys

print("Imports completados.")

---
## Celda 3 — Clase `ConsolaRouter`

Esta clase permite acceder al Router 2 (cat8k-pod02) cuya consola serial solo es accesible
desde `localhost:2223` del DevBox.

**Flujo:**
```
Tu máquina  →(SSH)→  DevBox (10.10.20.50)  →(telnet)→  Router 2 consola
```

**Por qué es necesario:** El socket QEMU está enlazado a `127.0.0.1` del DevBox — no accesible externamente.

**Por qué matamos sesiones previas:** QEMU solo acepta **una conexión simultánea** en el socket de consola. Si hay una sesión telnet activa, la nueva conexión queda colgada en `Trying 127.0.0.1...`.

In [ ]:
class ConsolaRouter:
    """
    Conecta a un router Cat8kv via la consola serial del DevBox.
    Flujo: SSH al DevBox → telnet localhost:<puerto_consola>
    Expone la misma interfaz que Netmiko para compatibilidad.
    """

    PROMPT_EXEC   = ['#']
    PROMPT_AUTH   = ['Username', 'username', 'Password', 'password', '#', '>']
    PROMPT_CONFIG = ['(config']

    def __init__(self, devbox_info, console_port, router_user, router_pass):
        print(f"    [>] SSH al DevBox ({devbox_info['host']})...")
        self.ssh = paramiko.SSHClient()
        self.ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
        self.ssh.connect(
            hostname=devbox_info['host'],
            username=devbox_info['username'],
            password=devbox_info['password'],
            look_for_keys=False,
            allow_agent=False
        )
        self.shell = self.ssh.invoke_shell(width=200, height=50)
        self._esperar_prompt(self.PROMPT_EXEC + ['$'], timeout=5)
        self._limpiar_buffer()

        # Liberar el puerto si hay una sesión telnet activa (QEMU acepta solo una)
        print(f"    [>] Liberando puerto {console_port} si hay sesión activa...")
        kill_cmd = (
            f"PID=$(ps aux | grep 'telnet localhost {console_port}' "
            f"| grep -v grep | awk '{{print $2}}') "
            f"&& [ -n \"$PID\" ] && kill $PID; sleep 1\n"
        )
        self.shell.send(kill_cmd)
        time.sleep(2)
        self._limpiar_buffer()

        print(f"    [>] Abriendo consola del router (puerto {console_port})...")
        self.shell.send(f'telnet localhost {console_port}\n')
        time.sleep(4)
        self._limpiar_buffer()

        # Enviar Enter para despertar la consola del router
        self.shell.send('\n')
        salida = self._esperar_prompt(self.PROMPT_AUTH, timeout=15)

        if 'Username' in salida or 'username' in salida:
            self.shell.send(f'{router_user}\n')
            salida = self._esperar_prompt(['Password', 'password', '#', '>'], timeout=5)

        if 'Password' in salida or 'password' in salida:
            self.shell.send(f'{router_pass}\n')
            salida = self._esperar_prompt(['#', '>'], timeout=10)

        # Si quedó en modo usuario, ejecutar enable
        if salida.rstrip().endswith('>'):
            self.shell.send('enable\n')
            salida = self._esperar_prompt(['Password', 'password', '#'], timeout=5)
            if 'Password' in salida or 'password' in salida:
                self.shell.send(f'{router_pass}\n')
                self._esperar_prompt(['#'], timeout=5)

        if '#' not in salida:
            self.shell.send('\n')
            self._esperar_prompt(['#'], timeout=5)

        self._limpiar_buffer()
        print(f"    [>] Sesión activa en el router.")

    def _esperar(self, segundos):
        time.sleep(segundos)

    def _leer(self):
        """Lee todo lo disponible en el buffer."""
        salida = b''
        time.sleep(0.3)
        while self.shell.recv_ready():
            salida += self.shell.recv(65535)
            time.sleep(0.1)
        return salida.decode('utf-8', errors='ignore')

    def _esperar_prompt(self, prompts, timeout=15):
        """Lee hasta encontrar uno de los prompts o agotar el timeout."""
        salida = ''
        inicio = time.time()
        while time.time() - inicio < timeout:
            if self.shell.recv_ready():
                chunk = self.shell.recv(4096).decode('utf-8', errors='ignore')
                salida += chunk
                if any(p in salida for p in prompts):
                    break
            else:
                time.sleep(0.2)
        return salida

    def _limpiar_buffer(self):
        self._leer()

    def send_config_set(self, commands):
        """Envía lista de comandos en modo configuración."""
        self.shell.send('configure terminal\n')
        self._esperar_prompt(self.PROMPT_CONFIG, timeout=5)
        for cmd in commands:
            self.shell.send(f'{cmd}\n')
            self._esperar(0.5)
        self.shell.send('end\n')
        self._esperar_prompt(self.PROMPT_EXEC, timeout=5)
        return self._leer()

    def send_command(self, command, read_timeout=15):
        """Envía un comando EXEC y retorna la salida."""
        self.shell.send(f'{command}\n')
        self._esperar_prompt(self.PROMPT_EXEC, timeout=read_timeout)
        return self._leer()

    def save_config(self):
        """Guarda la configuración (write memory)."""
        self.shell.send('write memory\n')
        salida = self._esperar_prompt(['[OK]', 'copy', '#'], timeout=10)
        if 'confirm' in salida.lower() or '[confirm]' in salida:
            self.shell.send('\n')
            self._esperar_prompt(['#'], timeout=10)
        return salida

    def disconnect(self):
        """Cierra la sesión y la conexión SSH al DevBox."""
        try:
            self.shell.send('exit\n')
            self._esperar(0.5)
        except Exception:
            pass
        self.ssh.close()

print("Clase ConsolaRouter definida.")

---
## Celda 4 — Configuración del DevBox y dispositivos

El diccionario `devices` usa dos tipos:
- `'tipo': 'netmiko'` → conexión SSH con Netmiko (Switch y Router 1)
- `'tipo': 'consola'` → túnel SSH via DevBox (Router 2)

In [ ]:
# Credenciales del DevBox (punto de entrada para Router 2)
DEVBOX = {
    'host': '10.10.20.50',
    'username': 'developer',
    'password': 'C1sco12345',
}

# Diccionario de dispositivos
devices = {
    'switch_nexus': {
        'tipo': 'netmiko',
        'params': {
            'device_type': 'cisco_nxos',   # Nexus OS
            'host': '10.10.20.40',
            'username': 'admin',
            'password': 'RG!_Yw200',
            'global_delay_factor': 2,
        }
    },
    'router1': {
        'tipo': 'netmiko',
        'params': {
            'device_type': 'cisco_ios',    # IOS XE
            'host': '10.10.20.48',         # Cat8k-1 — SSH directo confirmado
            'username': 'developer',
            'password': 'C1sco12345',
            'global_delay_factor': 2,
        }
    },
    'router2': {
        'tipo': 'consola',                 # Cat8k-2 — via consola DevBox
        'console_port': 2223,
        'router_user': 'developer',
        'router_pass': 'C1sco12345',
    }
}

print("Dispositivos configurados:")
for nombre, cfg in devices.items():
    tipo = cfg['tipo']
    if tipo == 'netmiko':
        print(f"  {nombre}: SSH directo a {cfg['params']['host']}")
    else:
        print(f"  {nombre}: consola via DevBox puerto {cfg['console_port']}")

---
## Celda 5 — Comandos de configuración

Los comandos están organizados por dispositivo. Cada lista es enviada en modo `configure terminal`.

> **Nota sobre Router 2:** Solo tiene `GigabitEthernet1` (una sola NIC en QEMU). Se usa `GigabitEthernet1.10` en lugar de `GigabitEthernet2.10`.

In [ ]:
config_commands = {
    # ── Switch Nexus 9K ────────────────────────────────────────────
    'switch_nexus': [
        'banner motd #ADVERTENCIA: Acceso autorizado únicamente a personal de TI#',
        'vlan 10',
        'name VLAN_COMPARTIDA',
        'exit',
        'interface Ethernet1/1',
        'switchport',
        'switchport mode trunk',
        'switchport trunk allowed vlan 10',
        'no shutdown',
        'interface Ethernet1/2',
        'switchport',
        'switchport mode trunk',
        'switchport trunk allowed vlan 10',
        'no shutdown',
    ],

    # ── Router 1 — Cat8k-1 (10.10.20.48) ──────────────────────────
    # Tiene GigabitEthernet1 (mgmt), GigabitEthernet2, GigabitEthernet3
    # Usamos GigabitEthernet2.10 para la subinterfaz VLAN 10
    'router1': [
        'banner motd #ADVERTENCIA: Acceso autorizado únicamente a personal de TI#',
        'service password-encryption',    # Encripta contraseñas en el config
        'no ip domain-lookup',            # Evita que IOS intente resolver typos como DNS
        'ip ssh version 2',               # Forzar SSH v2 (más seguro)
        'ip ssh authentication-retries 3',
        'ip ssh time-out 60',
        'line vty 0 4',
        'exec-timeout 5 0',               # Cierra sesión inactiva en 5 minutos
        'transport input ssh',            # Solo SSH, no telnet en VTY
        'exit',
        'interface GigabitEthernet2',
        'no shutdown',
        'interface GigabitEthernet2.10',
        'encapsulation dot1Q 10',         # Etiqueta VLAN 10 en tramas
        'ip address 192.168.10.1 255.255.255.0',
        'no shutdown',
    ],

    # ── Router 2 — Cat8k-2 / cat8k-pod02 (via DevBox consola) ─────
    # Solo tiene GigabitEthernet1 (una NIC en QEMU c8k5_vm)
    # Usamos GigabitEthernet1.10 como subinterfaz
    'router2': [
        'service password-encryption',
        'no ip domain-lookup',
        'interface GigabitEthernet1',
        'no shutdown',
        'interface GigabitEthernet1.10',
        'encapsulation dot1Q 10',
        'ip address 192.168.10.2 255.255.255.0',
        'no shutdown',
    ],
}

print("Comandos de configuración cargados:")
for dispositivo, cmds in config_commands.items():
    print(f"  {dispositivo}: {len(cmds)} comandos")

---
## Celda 6 — Funciones de conexión y envío

La función `conectar()` decide el método según `devices[name]['tipo']`:
- `'netmiko'` → `ConnectHandler` (Netmiko)
- `'consola'` → `ConsolaRouter` (Paramiko + telnet via DevBox)

In [ ]:
def conectar(device_name):
    """Establece conexión con el dispositivo según su tipo."""
    dev = devices[device_name]

    if dev['tipo'] == 'netmiko':
        host = dev['params']['host']
        print(f"\n[*] Conectando a {host} ({device_name}) via SSH...")
        try:
            conexion = ConnectHandler(**dev['params'])
            print(f"[+] Conexión establecida con {device_name}.")
            return conexion
        except Exception as e:
            print(f"[-] Error conectando a {device_name}: {e}")
            return None

    else:  # consola via DevBox
        print(f"\n[*] Conectando a {device_name} via consola del DevBox...")
        try:
            conexion = ConsolaRouter(
                DEVBOX,
                dev['console_port'],
                dev['router_user'],
                dev['router_pass']
            )
            print(f"[+] Conexión establecida con {device_name}.")
            return conexion
        except Exception as e:
            print(f"[-] Error conectando a {device_name}: {e}")
            return None


def enviar_comandos(conexion, device_name):
    """Envía comandos de configuración y guarda la configuración."""
    print(f"[*] Enviando comandos a {device_name}...")
    try:
        conexion.send_config_set(config_commands[device_name])
        conexion.save_config()  # write memory
        print(f"[+] Configuración aplicada y guardada en {device_name}.")
    except Exception as e:
        print(f"[-] Error enviando comandos a {device_name}: {e}")


print("Funciones conectar() y enviar_comandos() definidas.")

---
## Celda 7 — Configurar solo el Switch Nexus 9K

> Requiere VPN activa y conectividad a `10.10.20.40`

In [ ]:
conexion_sw = conectar('switch_nexus')

if conexion_sw:
    enviar_comandos(conexion_sw, 'switch_nexus')
    conexion_sw.disconnect()
    print("[*] Conexión cerrada: switch_nexus")
else:
    print("[-] No se pudo configurar el switch. Verificar VPN y credenciales.")

---
## Celda 8 — Configurar Router 1 (Cat8k-1)

> Conexión SSH directa a `10.10.20.48`

In [ ]:
conexion_r1 = conectar('router1')

if conexion_r1:
    enviar_comandos(conexion_r1, 'router1')
    conexion_r1.disconnect()
    print("[*] Conexión cerrada: router1")
else:
    print("[-] No se pudo configurar router1.")

---
## Celda 9 — Configurar Router 2 (Cat8k-2 via DevBox)

> Túnel SSH: máquina local → DevBox (10.10.20.50) → telnet localhost:2223

In [ ]:
conexion_r2 = conectar('router2')

if conexion_r2:
    enviar_comandos(conexion_r2, 'router2')
    print("[*] Configuración aplicada. Guardando referencia para el ping...")
    # No desconectar aún — se usa para el ping en la siguiente celda
else:
    print("[-] No se pudo configurar router2.")

---
## Celda 10 — Validación: Ping de Router 2 a Router 1

Confirma que la subinterfaz `GigabitEthernet1.10` (192.168.10.2) alcanza a `192.168.10.1` via la VLAN 10 troncal del switch.

In [ ]:
if conexion_r2:
    print("Esperando 5 segundos para convergencia de red...")
    time.sleep(5)

    print("Enviando ping desde Router 2 (192.168.10.2) hacia Router 1 (192.168.10.1)...")
    resultado = conexion_r2.send_command(
        "ping 192.168.10.1 source GigabitEthernet1.10",
        read_timeout=15
    )

    print("\n--- Resultado del Ping ---")
    print(resultado)
    print("--------------------------")

    conexion_r2.disconnect()
    print("[*] Conexión cerrada: router2")
else:
    print("[-] No hay conexión activa con router2.")

---
## Celda 11 — Ejecución completa (equivalente a opción 3 del script)

Esta celda ejecuta todo el flujo en una sola corrida:
Switch → Router 1 → Router 2 → Ping de validación

In [ ]:
print("=" * 50)
print("DESPLIEGUE COMPLETO DE INFRAESTRUCTURA")
print("=" * 50)

device_list = ['switch_nexus', 'router1', 'router2']
conexiones = {}

# Conectar y configurar cada dispositivo
for device_name in device_list:
    conexion = conectar(device_name)
    if conexion:
        enviar_comandos(conexion, device_name)
        conexiones[device_name] = conexion
    else:
        print(f"[-] No se pudo configurar {device_name}.")

# Validación con ping
if 'router2' in conexiones:
    print("\n[*] Esperando 5 segundos para convergencia...")
    time.sleep(5)
    print("Ping desde R2 (192.168.10.2) → R1 (192.168.10.1)...")
    resultado = conexiones['router2'].send_command(
        "ping 192.168.10.1 source GigabitEthernet1.10",
        read_timeout=15
    )
    print("\n--- Resultado del Ping ---")
    print(resultado)
    print("--------------------------")

# Cerrar todas las conexiones
for device_name, conexion in conexiones.items():
    conexion.disconnect()
    print(f"[*] Conexion cerrada: {device_name}")

print("\nDespliegue finalizado.")

---
## Celda 12 — Verificación manual del estado

Consulta el estado actual de las interfaces en cada dispositivo sin modificar nada.

In [ ]:
print("=== Verificación de estado ===")

# Switch
sw = conectar('switch_nexus')
if sw:
    print("\n--- Switch Nexus 9K ---")
    print(sw.send_command("show vlan brief"))
    print(sw.send_command("show interfaces trunk"))
    sw.disconnect()

# Router 1
r1 = conectar('router1')
if r1:
    print("\n--- Router 1 (10.10.20.48) ---")
    print(r1.send_command("show ip interface brief"))
    r1.disconnect()

# Router 2
r2 = conectar('router2')
if r2:
    print("\n--- Router 2 (cat8k-pod02) ---")
    print(r2.send_command("show ip interface brief"))
    r2.disconnect()